# 生产环境中的 AutoGen 智能体:可观测性与评估

在本教程中,我们将学习如何**监控 [Autogen 智能体](https://github.com/microsoft/autogen) 的内部步骤(追踪)**并使用 [Langfuse](https://langfuse.com) **评估其性能**。

本指南涵盖了团队用于快速可靠地将智能体投入生产环境的**在线**和**离线**评估指标。

**为什么 AI 智能体评估很重要:**
- 当任务失败或产生次优结果时调试问题
- 实时监控成本和性能
- 通过持续反馈提高可靠性和安全性


## 步骤 1:设置环境变量

通过注册 [Langfuse Cloud](https://cloud.langfuse.com/) 或[自托管 Langfuse](https://langfuse.com/self-hosting) 获取您的 Langfuse API 密钥。

_**注意:** 自托管用户可以使用 [Terraform 模块](https://langfuse.com/self-hosting/azure) 在 Azure 上部署 Langfuse。或者,您可以使用 [Helm chart](https://langfuse.com/self-hosting/kubernetes-helm) 在 Kubernetes 上部署 Langfuse。_

In [ ]:
import os

# 从项目设置页面获取项目的密钥: https://cloud.langfuse.com
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..." 
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..." 
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com" # 🇪🇺 欧洲区域
# os.environ["LANGFUSE_HOST"] = "https://us.cloud.langfuse.com" # 🇺🇸 美国区域

In [ ]:
!pip install langfuse openlit

设置好环境变量后,我们现在可以初始化 Langfuse 客户端。`get_client()` 使用环境变量中提供的凭据初始化 Langfuse 客户端。

In [ ]:
from langfuse import Langfuse
 
# 过滤掉 Autogen OpenTelemetry 追踪
langfuse = Langfuse(
    blocked_instrumentation_scopes=["autogen SingleThreadedAgentRuntime"]
)
 
# 验证连接
if langfuse.auth_check():
    print("Langfuse 客户端已通过身份验证并准备就绪!")
else:
    print("身份验证失败。请检查您的凭据和主机。")

## 步骤 2:初始化 OpenLit 检测

现在,我们初始化 [OpenLit](https://github.com/openlit/openlit) 检测。OpenLit 自动捕获 AutoGen 操作并将 OpenTelemetry (OTel) 追踪导出到 Langfuse。

In [ ]:
import openlit
 
# 初始化 OpenLIT 检测。disable_batch 标志设置为 true 以立即处理追踪。
openlit.init(tracer=langfuse._otel_tracer, disable_batch=True, disabled_instrumentors=["mistral"])

## 步骤 3:运行您的智能体

现在我们设置一个多轮智能体来测试我们的检测。

In [ ]:
import os

from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.azure import AzureAIChatCompletionClient
from azure.core.credentials import AzureKeyCredential
from autogen_agentchat.base import TaskResult

from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat

In [ ]:
client = AzureAIChatCompletionClient(
    model="gpt-4o-mini",
    endpoint="https://models.inference.ai.azure.com",
    # 要对模型进行身份验证,您需要在 GitHub 设置中生成个人访问令牌 (PAT)。
    # 按照此处的说明创建您的 PAT 令牌: https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens
    credential=AzureKeyCredential(os.environ["GITHUB_TOKEN"]),
    model_info={
        "json_output": True,
        "function_calling": True,
        "vision": True,
        "family": "unknown",
        "structured_output": False
    },
)

In [ ]:
# 🍴 智能体 1 – 每轮提出一个健康餐点想法
meal_planner_agent = AssistantAgent(
    "meal_planner_agent",
    model_client=client,
    description="一位经验丰富的膳食规划教练,建议均衡的餐点。",
    system_message="""
    您是一位拥有十年经验的膳食规划助手,帮助忙碌的人准备餐点。
    目标:根据用户的上下文提出最佳的单个餐点(早餐、午餐或晚餐)。
    每个回复必须仅包含一个完整的餐点想法(标题 + 非常简短的成分列表)——不要有其他内容。
    保持简洁:跳过问候、闲聊和填充语。
    """,
)

# 🥗 智能体 2 – 检查营养质量和多样性
nutritionist_agent = AssistantAgent(
    "nutritionist_agent",
    model_client=client,
    description="一位注册营养师,确保餐点符合营养标准。",
    system_message="""
    您是一位专注于全食物、宏观均衡饮食的营养师。
    评估 meal_planner_agent 的推荐。
    如果餐点营养合理、多样性足够且份量适当,请回复 'APPROVE'。
    否则,提供高级指导以改进它(例如 '添加植物性蛋白质')——不要提供完整的替代食谱。
    """,
)

In [ ]:
# ✅ 一旦营养师说 APPROVE,聊天就会停止
termination = TextMentionTermination("APPROVE")

# 🔄 在两个智能体之间轮流对话,直到终止
team = RoundRobinGroupChat(
    [meal_planner_agent, nutritionist_agent],
    termination_condition=termination,
)

# 示例启动
user_input = "I'm looking for a quick, delicious dinner I can prep after work. I have 30 minutes and minimal clean-up is ideal."

In [ ]:
with langfuse.start_as_current_span(name="create_meal_plan") as span:
    async for message in team.run_stream(task=user_input):
        if isinstance(message, TaskResult):
            print("Stop Reason:", message.stop_reason)
        else:
            print(message)

    span.update_trace(
        input=user_input,
        output=message.stop_reason,
    )

# 将追踪刷新到 Langfuse,适用于 Jupyter Notebooks 等短期环境
langfuse.flush()

### 追踪结构

Langfuse 记录一个**追踪**,其中包含**跨度**,代表智能体逻辑的每个步骤。在这里,追踪包含整体智能体运行以及以下子跨度:
- 膳食规划智能体
- 营养师智能体

您可以检查这些以准确了解时间花费在哪里、使用了多少令牌等:

![Langfuse 中的追踪树](https://langfuse.com/images/cookbook/example-autogen-evaluation/trace-tree.png)

_[追踪链接](https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/dac2b33e7cd709e685ccf86a137ecc64)_ 

## 在线评估

在线评估是指在实时、真实世界环境中评估智能体,即在生产环境中的实际使用期间。这涉及监控智能体在真实用户交互上的性能并持续分析结果。

### 生产环境中要跟踪的常见指标

1. **成本** — 检测捕获令牌使用情况,您可以通过为每个令牌分配价格将其转换为近似成本。
2. **延迟** — 观察完成每个步骤或整个运行所需的时间。
3. **用户反馈** — 用户可以提供直接反馈(点赞/点踩)以帮助改进或纠正智能体。
4. **LLM-as-a-Judge** — 使用单独的 LLM 近实时评估智能体的输出(例如,检查毒性或正确性)。

下面,我们展示这些指标的示例。

#### 1. 成本

以下是显示 `gpt-4o-mini` 调用使用情况的截图。这对于查看成本高昂的步骤并优化智能体非常有用。

![成本](https://langfuse.com/images/cookbook/example-autogen-evaluation/gpt-4o-costs.png) 

_[追踪链接](https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/dac2b33e7cd709e685ccf86a137ecc64)_

#### 2. 延迟

我们还可以看到完成每个步骤需要多长时间。在下面的示例中,整个运行大约需要 3 秒,您可以按步骤细分。这有助于您识别瓶颈并优化智能体。

![延迟](https://langfuse.com/images/cookbook/example-autogen-evaluation/agent-latency.png) 

_[追踪链接](https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/dac2b33e7cd709e685ccf86a137ecc64?display=timeline)_

#### 3. 用户反馈

如果您的智能体嵌入到用户界面中,您可以记录直接的用户反馈(例如聊天 UI 中的点赞/点踩)。

In [ ]:
from langfuse import get_client
 
langfuse = get_client()
 
# 选项 1: 使用上下文管理器生成的跨度对象
with langfuse.start_as_current_span(
    name="autogen-request-user-feedback-1") as span:
    
    async for message in team.run_stream(task="Create a meal with potatoes"):
            if isinstance(message, TaskResult):
                print("Stop Reason:", message.stop_reason)
            else:
                print(message)    
 
    # 使用跨度对象进行评分
    span.score_trace(
        name="user-feedback",
        value=1,
        data_type="NUMERIC",
        comment="This was delicious, thank you"
    )
 
# 选项 2: 如果仍在上下文中,使用 langfuse.score_current_trace()
with langfuse.start_as_current_span(name="autogen-request-user-feedback-2") as span:
    # ... Autogen 执行 ...

    async for message in team.run_stream(task="I am allergic to gluten."):
            if isinstance(message, TaskResult):
                print("Stop Reason:", message.stop_reason)
            else:
                print(message)    
 
    # 使用当前上下文进行评分
    langfuse.score_current_trace(
        name="user-feedback",
        value=1,
        data_type="NUMERIC"
    )

In [ ]:
# 选项 3: 使用 create_score() 和追踪 ID(在上下文之外时)
langfuse.create_score(
    trace_id="predefined_trace_id",
    name="user-feedback",
    value=1,
    data_type="NUMERIC",
    comment="This was correct, thank you"
)

用户反馈随后在 Langfuse 中被捕获:

![用户反馈正在 Langfuse 中被捕获](https://langfuse.com/images/cookbook/example-autogen-evaluation/user-feedback.png) 

#### 4. 自动化 LLM-as-a-Judge 评分

LLM-as-a-Judge 是另一种自动评估智能体输出的方法。您可以设置单独的 LLM 调用来评估输出的正确性、毒性、风格或您关心的任何其他标准。

**工作流程**:
1. 您定义一个**评估模板**,例如"检查文本是否有毒"。
2. 您设置一个用作评判模型的模型;在这种情况下是通过 Azure 查询的 `gpt-4o-mini`。
2. 每次您的智能体生成输出时,您将该输出与模板一起传递给您的"评判"LLM。
3. 评判 LLM 响应一个评分或标签,您将其记录到您的可观测性工具中。

Langfuse 中的示例:

![LLM-as-a-Judge 评估器](https://langfuse.com/images/cookbook/example-autogen-evaluation/evaluator.png) 

In [ ]:
with langfuse.start_as_current_span(name="autogen-request-user-feedback-2") as span:

    async for message in team.run_stream(task="I am a picky eater and not sure if you find something for me."):
            if isinstance(message, TaskResult):
                print("Stop Reason:", message.stop_reason)
            else:
                print(message) 

    span.update_trace(
        input=user_input,
        output=message.stop_reason,
    )

langfuse.flush()

您可以看到此示例的答案被评判为"not toxic"(无毒)。

![LLM-as-a-Judge 评估分数](https://langfuse.com/images/cookbook/example-autogen-evaluation/llm-as-a-judge-score.png) 

#### 5. 可观测性指标概览

所有这些指标都可以在仪表板中一起可视化。这使您能够快速查看智能体在许多会话中的表现,并帮助您随时间跟踪质量指标。

![可观测性指标概览](https://langfuse.com/images/cookbook/example-autogen-evaluation/dashboard.png) 

## 离线评估

在线评估对于实时反馈至关重要,但您还需要**离线评估**——在开发之前或开发期间的系统检查。这有助于在将更改投入生产之前保持质量和可靠性。

### 数据集评估

在离线评估中,您通常:
1. 拥有一个基准数据集(包含提示和预期输出对)
2. 在该数据集上运行您的智能体
3. 将输出与预期结果进行比较,或使用额外的评分机制

下面,我们使用 [q&a-dataset](https://huggingface.co/datasets/junzhang1207/search-dataset) 演示这种方法,该数据集包含问题和预期答案。

In [ ]:
import pandas as pd
from datasets import load_dataset
 
# 从 Hugging Face 获取 search-dataset
